# Post-Hoc DyT in Pretrained Transformer Encoders

Companion notebook for the 6.s058 final paper. Repo: [zhizhen-kyle-luo/6s058-final-project](https://github.com/zhizhen-kyle-luo/6s058-final-project).

## 1. Install

In [1]:
!pip install -q "transformers==4.46.3" "datasets>=3.2.0" "fsspec>=2025.3.0" "evaluate==0.4.3" "accelerate==1.1.1" "timm==1.0.11"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 131.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 100.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 111.7 MB/s eta 0:00:00


In [2]:
import torch, transformers, datasets, accelerate, timm
print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("datasets    ", datasets.__version__)
print("accelerate  ", accelerate.__version__)
print("timm        ", timm.__version__)
print("cuda        ", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


torch        2.10.0+cu128
transformers 4.46.3
datasets     4.0.0
accelerate   1.1.1
timm         1.0.11
cuda         True NVIDIA A100-SXM4-80GB


## 2. Repo + Drive

In [23]:
import os, pathlib

try:
    from google.colab import drive
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    try:
        drive.mount("/content/drive", force_remount=False)
        OUTPUTS_DIR = pathlib.Path("/content/drive/MyDrive/6s058-final-project")
    except Exception as e:
        print(f"Drive mount failed ({e}); using temporary /content outputs")
        OUTPUTS_DIR = pathlib.Path("/content/6s058-final-project")
else:
    OUTPUTS_DIR = pathlib.Path("./final_paper")

RUNS_DIR = OUTPUTS_DIR / "runs"
FIGURES_DIR = OUTPUTS_DIR / "figures"

print("RUNS_DIR    =", RUNS_DIR)
print("FIGURES_DIR =", FIGURES_DIR)
!nvidia-smi -L 2>/dev/null || echo "no nvidia-smi"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RUNS_DIR    = /content/drive/MyDrive/6s058-final-project/runs
FIGURES_DIR = /content/drive/MyDrive/6s058-final-project/figures
GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-670c9d6f-911c-e3ff-b78a-0f8be9b10d0c)


## 3. Config

In [4]:
from dataclasses import dataclass, asdict

@dataclass
class Config:
    task: str = "segmentation"   # "segmentation" or "classification"
    variant: str = "ln"          # "ln" or "dyt_full"
    run_name: str = ""
    use_amp: bool = True

    iters: int = 5000
    batch: int = 4
    crop: int = 512
    lr: float = 6e-5
    weight_decay: float = 0.01
    warmup_iters: int = 200
    eval_every: int = 500
    seed: int = 42
    pretrained: str = "nvidia/mit-b0"
    num_classes: int = 150

    # Segmentation-only settings; classification keeps placeholder values for a shared schema.
    ignore_index: int = 255
    subset_train: int = 2000
    subset_val: int = 500

    # DyT settings; used only when variant == "dyt_full".
    dyt_alpha_init: float = 0.5
    dyt_preserve_ln_affine: bool = False


def cfg_classification(**overrides):
    base = dict(
        task="classification",
        iters=3000,
        batch=64,
        crop=224,
        lr=5e-5,
        weight_decay=0.01,
        warmup_iters=200,
        eval_every=500,
        pretrained="google/vit-base-patch16-224",
        num_classes=100,
        ignore_index=-100,
    )
    base.update(overrides)
    return Config(**base)


def cfg_segmentation(**overrides):
    return Config(**overrides)

## 4. DyT module + LayerNorm replacement

In [5]:
import torch
import torch.nn as nn


class DyT(nn.Module):
    # Zhu et al. arXiv:2503.10622, Algorithm 1.
    # DyT(x) = gamma * tanh(alpha * x) + beta with learnable scalar alpha and per-channel gamma, beta.
    def __init__(self, normalized_shape, alpha_init: float = 0.5):
        super().__init__()
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape,)
        self.normalized_shape = tuple(normalized_shape)
        self.alpha = nn.Parameter(torch.tensor(float(alpha_init)))
        self.gamma = nn.Parameter(torch.ones(self.normalized_shape))
        self.beta  = nn.Parameter(torch.zeros(self.normalized_shape))

    def forward(self, x):
        # NCHW disambiguation: a 4D tensor whose channel dim matches normalized_shape is (B,C,H,W).
        if x.dim() == 4 and len(self.normalized_shape) == 1 and x.shape[1] == self.normalized_shape[0]:
            return self.gamma.view(1, -1, 1, 1) * torch.tanh(self.alpha * x) + self.beta.view(1, -1, 1, 1)
        if x.shape[-len(self.normalized_shape):] == self.normalized_shape:
            return self.gamma * torch.tanh(self.alpha * x) + self.beta
        raise RuntimeError(f"DyT shape mismatch: input {tuple(x.shape)} vs normalized_shape {self.normalized_shape}")


def _set_module(root, dotted, new):
    parts = dotted.split(".")
    parent = root
    for p in parts[:-1]:
        parent = getattr(parent, p)
    setattr(parent, parts[-1], new)


def _auto_backbone_prefix(model):
# Use "vit." so the final pre-classifier vit.layernorm is replaced too.
# SegFormer's decode head uses BatchNorm, so LayerNorm replacement stays in segformer.encoder.
    cls = model.__class__.__name__
    if "Segformer" in cls: return "segformer.encoder."
    if "ViT" in cls: return "vit."
    raise ValueError(f"unknown model class: {cls}; pass backbone_prefix explicitly")


def replace_ln_with_dyt(model, alpha_init=0.5, preserve_affine=False, backbone_prefix=None, verbose=True):
    if backbone_prefix is None:
        backbone_prefix = _auto_backbone_prefix(model)
    replaced = []
    for name, module in list(model.named_modules()):
        if not isinstance(module, nn.LayerNorm): continue
        if not name.startswith(backbone_prefix): continue
        params = list(module.parameters())
        device = params[0].device if params else torch.device("cpu")
        dtype = params[0].dtype if params else torch.float32
        new = DyT(module.normalized_shape, alpha_init=alpha_init).to(device=device, dtype=dtype)
        if preserve_affine:
            # Copy pretrained LN affine (weight, bias) into DyT (gamma, beta) so the activation
            # contract the downstream weights expect is preserved at the moment of swap.
            with torch.no_grad():
                new.gamma.copy_(module.weight)
                new.beta.copy_(module.bias)
        _set_module(model, name, new)
        replaced.append(name)
    remaining = [n for n, m in model.named_modules() if isinstance(m, nn.LayerNorm)]
    if verbose:
        print(f"[replace_ln_with_dyt] preserve_affine={preserve_affine} replaced {len(replaced)} LayerNorms under {backbone_prefix!r}")
        if remaining:
            preview = remaining[:3] + (["..."] if len(remaining) > 3 else [])
            print(f"[replace_ln_with_dyt] WARNING: {len(remaining)} LayerNorm(s) remain outside the prefix: {preview}")
    return replaced


def _check_affine_preserve():
    ln = nn.LayerNorm(32)
    with torch.no_grad():
        ln.weight.uniform_(0.5, 1.5); ln.bias.uniform_(-1.0, 1.0)
    container = nn.Sequential()
    container.add_module("vit", nn.Sequential())
    container.vit.add_module("layernorm", ln)
    rep = replace_ln_with_dyt(container, preserve_affine=True, backbone_prefix="vit.", verbose=False)
    new = container.vit.layernorm
    assert torch.allclose(new.gamma, ln.weight)
    assert torch.allclose(new.beta, ln.bias)
    assert len(rep) == 1
_check_affine_preserve()


## 5. Saturation hooks + α logger

In [6]:
from collections import defaultdict

class SaturationTracker:
    def __init__(self, model, threshold=0.95):
        self.threshold = threshold; self.values = defaultdict(list); self.handles = []
        for name, m in model.named_modules():
            if isinstance(m, DyT):
                self.handles.append(m.register_forward_hook(self._make_hook(name)))

    def _make_hook(self, name):
        def hook(module, inp, out):
            with torch.no_grad():
                pre = torch.tanh(module.alpha * inp[0])
                self.values[name].append((pre.abs() > self.threshold).float().mean().item())
        return hook

    def summary(self):
        return {k: float(sum(v) / max(len(v), 1)) for k, v in self.values.items()}

    def close(self):
        for h in self.handles: h.remove()


def collect_alphas(model):
    return {name: float(m.alpha.detach().cpu().item()) for name, m in model.named_modules() if isinstance(m, DyT)}


## 6. Data

In [7]:
import random
import numpy as np
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.transforms import functional as TF

# HF checkpoint preprocessing constants; these are not DyT-paper constants.
SEG_MEAN = (0.485, 0.456, 0.406)  # nvidia/mit-b0
SEG_STD  = (0.229, 0.224, 0.225)
CLS_MEAN = (0.5, 0.5, 0.5)        # google/vit-base-patch16-224
CLS_STD  = (0.5, 0.5, 0.5)
# zhoubolei/scene_parse_150 still ships a loading script on main; use a Parquet duplicate.
SEGMENTATION_DATASET = "merve/scene_parse_150"
CLASSIFICATION_DATASET = "uoft-cs/cifar100"
SEGMENTATION_COLUMNS = ("image", "annotation")


def _seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _check_segmentation_schema(ds):
    for split in ("train", "validation"):
        if split not in ds:
            raise KeyError(f"{SEGMENTATION_DATASET} missing split {split!r}; found {list(ds.keys())}")
        missing = set(SEGMENTATION_COLUMNS) - set(ds[split].column_names)
        if missing:
            raise KeyError(f"{SEGMENTATION_DATASET}/{split} missing columns {sorted(missing)}; found {ds[split].column_names}")
    print(f"{SEGMENTATION_DATASET} schema OK", {split: ds[split].column_names for split in ("train", "validation")})


class ADE20KSubset(Dataset):
    def __init__(self, hf_split, indices, crop, train):
        self.ds = hf_split; self.indices = indices; self.crop = crop; self.train = train

    def __len__(self): return len(self.indices)

    def _remap(self, mask):
        # HF masks use 0 as ignore and 1..150 as classes; CE expects 0..149 plus ignore_index=255.
        m = mask.long()
        out = torch.full_like(m, 255)
        fg = m >= 1
        out[fg] = m[fg] - 1
        return out

    def __getitem__(self, i):
        ex = self.ds[int(self.indices[i])]
        img = ex["image"].convert("RGB"); msk = ex["annotation"]
        if self.train:
            scale = random.uniform(0.5, 2.0)
            new_size = (int(img.height * scale), int(img.width * scale))
            img = TF.resize(img, new_size, T.InterpolationMode.BILINEAR)
            msk = TF.resize(msk, new_size, T.InterpolationMode.NEAREST)
            pad_h = max(0, self.crop - img.height); pad_w = max(0, self.crop - img.width)
            if pad_h or pad_w:
                img = TF.pad(img, [0, 0, pad_w, pad_h], fill=0)
                msk = TF.pad(msk, [0, 0, pad_w, pad_h], fill=0)
            i0 = random.randint(0, img.height - self.crop)
            j0 = random.randint(0, img.width  - self.crop)
            img = TF.crop(img, i0, j0, self.crop, self.crop)
            msk = TF.crop(msk, i0, j0, self.crop, self.crop)
            if random.random() < 0.5:
                img = TF.hflip(img); msk = TF.hflip(msk)
            img = T.ColorJitter(0.4, 0.4, 0.4)(img)
        else:
            img = TF.resize(img, self.crop, T.InterpolationMode.BILINEAR)
            msk = TF.resize(msk, self.crop, T.InterpolationMode.NEAREST)
            img = TF.center_crop(img, self.crop)
            msk = TF.center_crop(msk, self.crop)
        img_t = TF.normalize(TF.to_tensor(img), SEG_MEAN, SEG_STD)
        msk_t = self._remap(torch.as_tensor(np.array(msk), dtype=torch.long))
        return img_t, msk_t


class CIFAR100Wrap(Dataset):
    def __init__(self, hf_split, crop, train):
        self.ds = hf_split; self.crop = crop; self.train = train

    def __len__(self): return len(self.ds)

    def __getitem__(self, i):
        ex = self.ds[int(i)]
        img = ex["img"].convert("RGB")
        label = int(ex["fine_label"])
        if self.train:
            img = TF.resize(img, self.crop, T.InterpolationMode.BILINEAR)
            img = T.RandomResizedCrop(self.crop, scale=(0.9, 1.0))(img)
            if random.random() < 0.5:
                img = TF.hflip(img)
        else:
            img = TF.resize(img, self.crop, T.InterpolationMode.BILINEAR)
            img = TF.center_crop(img, self.crop)
        img_t = TF.normalize(TF.to_tensor(img), CLS_MEAN, CLS_STD)
        return img_t, torch.tensor(label, dtype=torch.long)


def build_loaders(cfg):
    _seed_everything(cfg.seed)
    if cfg.task == "segmentation":
        ds = load_dataset(SEGMENTATION_DATASET)
        _check_segmentation_schema(ds)
        # ADE20K is subsetted for compute; set subset_train/subset_val to full split sizes for other budgets.
        rng = np.random.RandomState(cfg.seed)
        train_idx = rng.choice(len(ds["train"]), size=cfg.subset_train, replace=False)
        val_idx   = rng.choice(len(ds["validation"]), size=cfg.subset_val, replace=False)
        train_ds = ADE20KSubset(ds["train"], train_idx, cfg.crop, train=True)
        val_ds   = ADE20KSubset(ds["validation"], val_idx, cfg.crop, train=False)
    elif cfg.task == "classification":
        ds = load_dataset(CLASSIFICATION_DATASET)
        # CIFAR-100 has only train+test; we use HF "test" as validation
        train_ds = CIFAR100Wrap(ds["train"], cfg.crop, train=True)
        val_ds   = CIFAR100Wrap(ds["test"],  cfg.crop, train=False)
    else:
        raise ValueError(cfg.task)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch, shuffle=True,  num_workers=2, drop_last=True,  pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=cfg.batch, shuffle=False, num_workers=2, drop_last=False, pin_memory=True)
    return train_loader, val_loader


## 7. Build model

In [8]:
from transformers import SegformerForSemanticSegmentation, ViTForImageClassification

def build_model(cfg):
    if cfg.task == "segmentation":
        model = SegformerForSemanticSegmentation.from_pretrained(
            cfg.pretrained, num_labels=cfg.num_classes, ignore_mismatched_sizes=True,
        )
    elif cfg.task == "classification":
        model = ViTForImageClassification.from_pretrained(
            cfg.pretrained, num_labels=cfg.num_classes, ignore_mismatched_sizes=True,
        )
    else:
        raise ValueError(cfg.task)
    if cfg.variant == "ln":
        return model, []
    if cfg.variant == "dyt_full":
        replaced = replace_ln_with_dyt(
            model,
            alpha_init=cfg.dyt_alpha_init,
            preserve_affine=cfg.dyt_preserve_ln_affine,
        )
        return model, replaced
    raise ValueError(cfg.variant)


## Smoke test

In [9]:
# ViT-B/16: 12 transformer blocks * 2 LayerNorms each + 1 final pre-classifier LayerNorm = 25.
# MiT-B0 (SegFormer): 4 patch-embedding LNs + 8 blocks * 2 LNs + 4 stage-final LNs + 6 sr_ratio>1 LNs = 30.
EXPECTED_REPLACEMENTS = {"classification": 25, "segmentation": 30}

def smoke_models():
    for cfg in [
        cfg_segmentation(task="segmentation", variant="dyt_full", run_name="smoke_seg",
                         subset_train=4, subset_val=4, batch=2, crop=128),
        cfg_classification(variant="dyt_full", run_name="smoke_vit", batch=2, iters=1),
    ]:
        loader, _ = build_loaders(cfg)
        xb, yb = next(iter(loader))
        model, replaced = build_model(cfg)
        expected = EXPECTED_REPLACEMENTS[cfg.task]
        assert len(replaced) == expected, f"{cfg.task}: expected {expected} replacements, got {len(replaced)}"
        remaining = [n for n, m in model.named_modules() if isinstance(m, nn.LayerNorm)]
        assert not remaining, f"{cfg.task}: {len(remaining)} LayerNorm(s) still present after replacement: {remaining[:3]}"
        out = model(pixel_values=xb).logits
        print(cfg.task, len(replaced), "replaced  out:", tuple(out.shape))
    print("smoke OK")

smoke_models()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/212M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/89.1M [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/442M [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([100]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([100, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[replace_ln_with_dyt] preserve_affine=False replaced 25 LayerNorms under 'vit.'
classification 25 replaced  out: (2, 100)
smoke OK


## 8. Train loop

In [10]:
import json, time, math


def _poly_lr(base, step, total, warmup, power=1.0):
    if step < warmup:
        return base * step / max(1, warmup)
    p = (step - warmup) / max(1, total - warmup)
    return base * (1 - p) ** power


def _cosine_lr(base, step, total, warmup):
    if step < warmup:
        return base * step / max(1, warmup)
    p = (step - warmup) / max(1, total - warmup)
    return base * 0.5 * (1.0 + math.cos(math.pi * p))


def _raise_incompatible_checkpoint(cfg, last_path, exc):
    original = str(exc)
    if "vit.layernorm.alpha" in original and "vit.layernorm.weight" in original:
        detail = (
            "This run was saved before the ViT final LayerNorm was included in the DyT swap. "
            "The current model has the 25-LayerNorm DyT layout, so it must start from a fresh run directory."
        )
    else:
        detail = "The saved checkpoint module layout or optimizer state does not match this config."
    raise RuntimeError(
        f"Cannot resume {last_path} for run_name={cfg.run_name!r}, task={cfg.task!r}, "
        f"variant={cfg.variant!r}. {detail}\n"
        f"Run clear_runs({cfg.run_name!r}) before training again, or choose a new run_name."
    ) from exc


@torch.no_grad()
def run_eval(model, loader, cfg, device, use_amp):
    model.eval()
    if cfg.task == "segmentation":
        cm = torch.zeros(cfg.num_classes, cfg.num_classes, dtype=torch.long, device=device)
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            with torch.amp.autocast("cuda", enabled=use_amp and device == "cuda"):
                out = model(pixel_values=xb).logits
                out = nn.functional.interpolate(out, size=yb.shape[-2:], mode="bilinear", align_corners=False)
            pred = out.argmax(1)
            valid = yb != cfg.ignore_index
            p = pred[valid]; t = yb[valid]
            idx = t * cfg.num_classes + p
            cm += torch.bincount(idx, minlength=cfg.num_classes * cfg.num_classes).reshape(cfg.num_classes, cfg.num_classes)
        cm = cm.cpu()
        inter = cm.diag().float()
        union = cm.sum(0).float() + cm.sum(1).float() - inter
        iou = torch.where(union > 0, inter / union, torch.full_like(inter, float("nan")))
        return {"primary": float(torch.nanmean(iou).item()), "metric_name": "miou"}
    elif cfg.task == "classification":
        correct = 0; total = 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            with torch.amp.autocast("cuda", enabled=use_amp and device == "cuda"):
                logits = model(pixel_values=xb).logits
            pred = logits.argmax(1)
            correct += (pred == yb).sum().item()
            total   += yb.numel()
        return {"primary": correct / max(1, total), "metric_name": "top1"}
    else:
        raise ValueError(cfg.task)


def train_one(cfg, runs_dir):
    assert cfg.run_name, "Config.run_name is required"
    _seed_everything(cfg.seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    use_amp = cfg.use_amp and device == "cuda"

    run_dir = runs_dir / cfg.run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "config.json").write_text(json.dumps(asdict(cfg), indent=2))

    train_loader, val_loader = build_loaders(cfg)
    model, replaced = build_model(cfg)
    model = model.to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    lr_fn = _cosine_lr if cfg.task == "classification" else _poly_lr

    start_iter = 0
    best_primary = -1.0
    last_path = run_dir / "last.pt"
    best_path = run_dir / "best.pt"
    if last_path.exists():
        # weights_only=False because the checkpoint stores optimizer + scaler state, not just tensors.
        ck = torch.load(last_path, map_location=device, weights_only=False)
        try:
            model.load_state_dict(ck["model"])
            optim.load_state_dict(ck["optim"])
        except (RuntimeError, ValueError) as exc:
            _raise_incompatible_checkpoint(cfg, last_path, exc)
        if "scaler" in ck and use_amp and ck["scaler"] is not None:
            scaler.load_state_dict(ck["scaler"])
        start_iter = ck["iter"]; best_primary = ck.get("best_primary", -1.0)
        print(f"resumed iter={start_iter} best={best_primary:.4f}")

    log_path = run_dir / "log.jsonl"
    log_f = open(log_path, "a")
    train_iter = iter(train_loader)
    sat = SaturationTracker(model) if any(isinstance(m, DyT) for m in model.modules()) else None

    t0 = time.time()
    model.train()
    for step in range(start_iter, cfg.iters):
        try: xb, yb = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader); xb, yb = next(train_iter)
        xb, yb = xb.to(device), yb.to(device)
        lr = lr_fn(cfg.lr, step, cfg.iters, cfg.warmup_iters)
        for g in optim.param_groups: g["lr"] = lr
        optim.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=use_amp):
            if cfg.task == "segmentation":
                out = model(pixel_values=xb).logits
                out = nn.functional.interpolate(out, size=yb.shape[-2:], mode="bilinear", align_corners=False)
                loss = nn.functional.cross_entropy(out, yb, ignore_index=cfg.ignore_index)
            elif cfg.task == "classification":
                out = model(pixel_values=xb).logits
                loss = nn.functional.cross_entropy(out, yb)
            else:
                raise ValueError(cfg.task)
        scaler.scale(loss).backward()
        scaler.step(optim); scaler.update()

        if (step + 1) % 50 == 0:
            print(f"[{cfg.run_name}] step {step+1}/{cfg.iters}  loss={loss.item():.4f}  lr={lr:.2e}  elapsed={time.time()-t0:.0f}s")
            log_f.write(json.dumps({"kind": "train", "step": step+1, "loss": float(loss.item()), "lr": lr}) + "\n"); log_f.flush()

        if (step + 1) % cfg.eval_every == 0 or (step + 1) == cfg.iters:
            metrics = run_eval(model, val_loader, cfg, device, use_amp)
            sat_summary = sat.summary() if sat is not None else {}
            print(f"[{cfg.run_name}] eval @ {step+1}: {metrics['metric_name']}={metrics['primary']:.4f}")
            log_f.write(json.dumps({"kind": "eval", "step": step+1, **metrics, "saturation": sat_summary}) + "\n"); log_f.flush()
            if metrics["primary"] > best_primary:
                best_primary = metrics["primary"]
                torch.save({"model": model.state_dict(), "iter": step+1, "primary": best_primary}, best_path)
            torch.save({
                "model": model.state_dict(), "optim": optim.state_dict(),
                "scaler": scaler.state_dict() if use_amp else None,
                "iter": step+1, "best_primary": best_primary,
            }, last_path)
            model.train()

    log_f.close()
    if sat is not None: sat.close()

    summary = {
        "run_name": cfg.run_name,
        "task": cfg.task,
        "variant": cfg.variant,
        "data_source": SEGMENTATION_DATASET if cfg.task == "segmentation" else CLASSIFICATION_DATASET,
        "iters": cfg.iters,
        "best_primary": best_primary,
        "metric_name": "miou" if cfg.task == "segmentation" else "top1",
        "num_replaced": len(replaced),
        "replaced_modules": replaced,
        "final_alphas": collect_alphas(model),
        "final_saturation": sat.summary() if sat is not None else {},
        "dyt_alpha_init": cfg.dyt_alpha_init,
        "dyt_preserve_ln_affine": cfg.dyt_preserve_ln_affine,
    }
    (run_dir / "summary.json").write_text(json.dumps(summary, indent=2))
    print("DONE", cfg.run_name, "best=", best_primary)
    return summary


## 9. Runs

`train_one` automatically resumes from `runs/<run_name>/last.pt` when it exists. Use the helper below before final clean reruns, or after changing model/data/training code in a way that makes old checkpoints incomparable.

In [11]:
import shutil

def clear_runs(*run_names, runs_dir=RUNS_DIR):
    for name in run_names:
        path = runs_dir / name
        if path.exists():
            shutil.rmtree(path); print("removed", path)
        else:
            print("missing", path)

# For a full clean rerun:
# clear_runs("seg_ln", "seg_dyt_full_fresh", "seg_dyt_full_affine",
#            "vit_ln", "vit_dyt_full_fresh", "vit_dyt_full_affine")


### Run `seg_ln` — segmentation, LayerNorm baseline (no replacement)

In [12]:
train_one(cfg_segmentation(task="segmentation", variant="ln", run_name="seg_ln"), RUNS_DIR)


merve/scene_parse_150 schema OK {'train': ['image', 'annotation', 'scene_category'], 'validation': ['image', 'annotation', 'scene_category']}


Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/mit-b0 and are newly initialized: ['decode_head.batch_norm.bias', 'decode_head.batch_norm.num_batches_tracked', 'decode_head.batch_norm.running_mean', 'decode_head.batch_norm.running_var', 'decode_head.batch_norm.weight', 'decode_head.classifier.bias', 'decode_head.classifier.weight', 'decode_head.linear_c.0.proj.bias', 'decode_head.linear_c.0.proj.weight', 'decode_head.linear_c.1.proj.bias', 'decode_head.linear_c.1.proj.weight', 'decode_head.linear_c.2.proj.bias', 'decode_head.linear_c.2.proj.weight', 'decode_head.linear_c.3.proj.bias', 'decode_head.linear_c.3.proj.weight', 'decode_head.linear_fuse.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/14.3M [00:00<?, ?B/s]

[seg_ln] step 50/5000  loss=4.9179  lr=1.47e-05  elapsed=6s
[seg_ln] step 100/5000  loss=4.5511  lr=2.97e-05  elapsed=9s
[seg_ln] step 150/5000  loss=4.1148  lr=4.47e-05  elapsed=13s
[seg_ln] step 200/5000  loss=3.4568  lr=5.97e-05  elapsed=16s
[seg_ln] step 250/5000  loss=2.7359  lr=5.94e-05  elapsed=19s
[seg_ln] step 300/5000  loss=3.3602  lr=5.88e-05  elapsed=22s
[seg_ln] step 350/5000  loss=2.7885  lr=5.81e-05  elapsed=25s
[seg_ln] step 400/5000  loss=2.9406  lr=5.75e-05  elapsed=28s
[seg_ln] step 450/5000  loss=2.7770  lr=5.69e-05  elapsed=31s
[seg_ln] step 500/5000  loss=3.0308  lr=5.63e-05  elapsed=34s
[seg_ln] eval @ 500: miou=0.0367
[seg_ln] step 550/5000  loss=1.9123  lr=5.56e-05  elapsed=43s
[seg_ln] step 600/5000  loss=2.6659  lr=5.50e-05  elapsed=47s
[seg_ln] step 650/5000  loss=2.1404  lr=5.44e-05  elapsed=50s
[seg_ln] step 700/5000  loss=2.3863  lr=5.38e-05  elapsed=53s
[seg_ln] step 750/5000  loss=2.3017  lr=5.31e-05  elapsed=56s
[seg_ln] step 800/5000  loss=1.7510  lr=

{'run_name': 'seg_ln',
 'task': 'segmentation',
 'variant': 'ln',
 'data_source': 'merve/scene_parse_150',
 'iters': 5000,
 'best_primary': 0.07649524509906769,
 'metric_name': 'miou',
 'num_replaced': 0,
 'replaced_modules': [],
 'final_alphas': {},
 'final_saturation': {},
 'dyt_alpha_init': 0.5,
 'dyt_preserve_ln_affine': False}

### Run `seg_dyt_full_fresh` — segmentation, DyT with paper-default init (γ=1, β=0, α₀=0.5)

In [13]:
train_one(cfg_segmentation(task="segmentation", variant="dyt_full", run_name="seg_dyt_full_fresh"), RUNS_DIR)


merve/scene_parse_150 schema OK {'train': ['image', 'annotation', 'scene_category'], 'validation': ['image', 'annotation', 'scene_category']}


Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/mit-b0 and are newly initialized: ['decode_head.batch_norm.bias', 'decode_head.batch_norm.num_batches_tracked', 'decode_head.batch_norm.running_mean', 'decode_head.batch_norm.running_var', 'decode_head.batch_norm.weight', 'decode_head.classifier.bias', 'decode_head.classifier.weight', 'decode_head.linear_c.0.proj.bias', 'decode_head.linear_c.0.proj.weight', 'decode_head.linear_c.1.proj.bias', 'decode_head.linear_c.1.proj.weight', 'decode_head.linear_c.2.proj.bias', 'decode_head.linear_c.2.proj.weight', 'decode_head.linear_c.3.proj.bias', 'decode_head.linear_c.3.proj.weight', 'decode_head.linear_fuse.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[replace_ln_with_dyt] preserve_affine=False replaced 30 LayerNorms under 'segformer.encoder.'
[seg_dyt_full_fresh] step 50/5000  loss=4.9939  lr=1.47e-05  elapsed=4s
[seg_dyt_full_fresh] step 100/5000  loss=4.7696  lr=2.97e-05  elapsed=7s
[seg_dyt_full_fresh] step 150/5000  loss=4.4044  lr=4.47e-05  elapsed=11s
[seg_dyt_full_fresh] step 200/5000  loss=4.3563  lr=5.97e-05  elapsed=14s
[seg_dyt_full_fresh] step 250/5000  loss=3.9945  lr=5.94e-05  elapsed=18s
[seg_dyt_full_fresh] step 300/5000  loss=4.2901  lr=5.88e-05  elapsed=21s
[seg_dyt_full_fresh] step 350/5000  loss=3.5998  lr=5.81e-05  elapsed=24s
[seg_dyt_full_fresh] step 400/5000  loss=3.7908  lr=5.75e-05  elapsed=28s
[seg_dyt_full_fresh] step 450/5000  loss=3.4724  lr=5.69e-05  elapsed=31s
[seg_dyt_full_fresh] step 500/5000  loss=3.7227  lr=5.63e-05  elapsed=35s
[seg_dyt_full_fresh] eval @ 500: miou=0.0082
[seg_dyt_full_fresh] step 550/5000  loss=3.5672  lr=5.56e-05  elapsed=44s
[seg_dyt_full_fresh] step 600/5000  loss=4.0541  l

{'run_name': 'seg_dyt_full_fresh',
 'task': 'segmentation',
 'variant': 'dyt_full',
 'data_source': 'merve/scene_parse_150',
 'iters': 5000,
 'best_primary': 0.01381948683410883,
 'metric_name': 'miou',
 'num_replaced': 30,
 'replaced_modules': ['segformer.encoder.patch_embeddings.0.layer_norm',
  'segformer.encoder.patch_embeddings.1.layer_norm',
  'segformer.encoder.patch_embeddings.2.layer_norm',
  'segformer.encoder.patch_embeddings.3.layer_norm',
  'segformer.encoder.block.0.0.layer_norm_1',
  'segformer.encoder.block.0.0.attention.self.layer_norm',
  'segformer.encoder.block.0.0.layer_norm_2',
  'segformer.encoder.block.0.1.layer_norm_1',
  'segformer.encoder.block.0.1.attention.self.layer_norm',
  'segformer.encoder.block.0.1.layer_norm_2',
  'segformer.encoder.block.1.0.layer_norm_1',
  'segformer.encoder.block.1.0.attention.self.layer_norm',
  'segformer.encoder.block.1.0.layer_norm_2',
  'segformer.encoder.block.1.1.layer_norm_1',
  'segformer.encoder.block.1.1.attention.self

### Run `seg_dyt_full_affine` — segmentation, DyT with affine-preserving init (γ←γ_LN, β←β_LN, α₀=0.5)

In [14]:
train_one(cfg_segmentation(task="segmentation", variant="dyt_full", run_name="seg_dyt_full_affine", dyt_preserve_ln_affine=True), RUNS_DIR)


merve/scene_parse_150 schema OK {'train': ['image', 'annotation', 'scene_category'], 'validation': ['image', 'annotation', 'scene_category']}


Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/mit-b0 and are newly initialized: ['decode_head.batch_norm.bias', 'decode_head.batch_norm.num_batches_tracked', 'decode_head.batch_norm.running_mean', 'decode_head.batch_norm.running_var', 'decode_head.batch_norm.weight', 'decode_head.classifier.bias', 'decode_head.classifier.weight', 'decode_head.linear_c.0.proj.bias', 'decode_head.linear_c.0.proj.weight', 'decode_head.linear_c.1.proj.bias', 'decode_head.linear_c.1.proj.weight', 'decode_head.linear_c.2.proj.bias', 'decode_head.linear_c.2.proj.weight', 'decode_head.linear_c.3.proj.bias', 'decode_head.linear_c.3.proj.weight', 'decode_head.linear_fuse.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[replace_ln_with_dyt] preserve_affine=True replaced 30 LayerNorms under 'segformer.encoder.'
[seg_dyt_full_affine] step 50/5000  loss=4.9512  lr=1.47e-05  elapsed=4s
[seg_dyt_full_affine] step 100/5000  loss=4.8777  lr=2.97e-05  elapsed=7s
[seg_dyt_full_affine] step 150/5000  loss=4.6206  lr=4.47e-05  elapsed=11s
[seg_dyt_full_affine] step 200/5000  loss=4.3770  lr=5.97e-05  elapsed=14s
[seg_dyt_full_affine] step 250/5000  loss=4.1020  lr=5.94e-05  elapsed=17s
[seg_dyt_full_affine] step 300/5000  loss=4.4308  lr=5.88e-05  elapsed=21s
[seg_dyt_full_affine] step 350/5000  loss=3.7598  lr=5.81e-05  elapsed=24s
[seg_dyt_full_affine] step 400/5000  loss=3.9554  lr=5.75e-05  elapsed=28s
[seg_dyt_full_affine] step 450/5000  loss=3.6681  lr=5.69e-05  elapsed=31s
[seg_dyt_full_affine] step 500/5000  loss=3.8492  lr=5.63e-05  elapsed=35s
[seg_dyt_full_affine] eval @ 500: miou=0.0050
[seg_dyt_full_affine] step 550/5000  loss=3.5622  lr=5.56e-05  elapsed=44s
[seg_dyt_full_affine] step 600/5000  lo

{'run_name': 'seg_dyt_full_affine',
 'task': 'segmentation',
 'variant': 'dyt_full',
 'data_source': 'merve/scene_parse_150',
 'iters': 5000,
 'best_primary': 0.009908029809594154,
 'metric_name': 'miou',
 'num_replaced': 30,
 'replaced_modules': ['segformer.encoder.patch_embeddings.0.layer_norm',
  'segformer.encoder.patch_embeddings.1.layer_norm',
  'segformer.encoder.patch_embeddings.2.layer_norm',
  'segformer.encoder.patch_embeddings.3.layer_norm',
  'segformer.encoder.block.0.0.layer_norm_1',
  'segformer.encoder.block.0.0.attention.self.layer_norm',
  'segformer.encoder.block.0.0.layer_norm_2',
  'segformer.encoder.block.0.1.layer_norm_1',
  'segformer.encoder.block.0.1.attention.self.layer_norm',
  'segformer.encoder.block.0.1.layer_norm_2',
  'segformer.encoder.block.1.0.layer_norm_1',
  'segformer.encoder.block.1.0.attention.self.layer_norm',
  'segformer.encoder.block.1.0.layer_norm_2',
  'segformer.encoder.block.1.1.layer_norm_1',
  'segformer.encoder.block.1.1.attention.se

### Run `vit_ln` — classification, LayerNorm baseline (soft sanity: top-1 ≥ 60%)

In [15]:
train_one(cfg_classification(variant="ln", run_name="vit_ln"), RUNS_DIR)


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([100]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([100, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[vit_ln] step 50/3000  loss=4.6090  lr=1.22e-05  elapsed=5s
[vit_ln] step 100/3000  loss=4.1310  lr=2.48e-05  elapsed=10s
[vit_ln] step 150/3000  loss=3.0479  lr=3.72e-05  elapsed=15s
[vit_ln] step 200/3000  loss=1.8869  lr=4.98e-05  elapsed=20s
[vit_ln] step 250/3000  loss=1.1674  lr=5.00e-05  elapsed=25s
[vit_ln] step 300/3000  loss=0.8564  lr=4.98e-05  elapsed=30s
[vit_ln] step 350/3000  loss=0.6423  lr=4.97e-05  elapsed=34s
[vit_ln] step 400/3000  loss=0.5558  lr=4.94e-05  elapsed=39s
[vit_ln] step 450/3000  loss=0.3512  lr=4.90e-05  elapsed=44s
[vit_ln] step 500/3000  loss=0.3475  lr=4.86e-05  elapsed=50s
[vit_ln] eval @ 500: top1=0.8870
[vit_ln] step 550/3000  loss=0.3785  lr=4.81e-05  elapsed=66s
[vit_ln] step 600/3000  loss=0.3198  lr=4.75e-05  elapsed=71s
[vit_ln] step 650/3000  loss=0.2157  lr=4.69e-05  elapsed=76s
[vit_ln] step 700/3000  loss=0.3743  lr=4.62e-05  elapsed=81s
[vit_ln] step 750/3000  loss=0.3937  lr=4.54e-05  elapsed=86s
[vit_ln] step 800/3000  loss=0.1061  lr

{'run_name': 'vit_ln',
 'task': 'classification',
 'variant': 'ln',
 'data_source': 'uoft-cs/cifar100',
 'iters': 3000,
 'best_primary': 0.9291,
 'metric_name': 'top1',
 'num_replaced': 0,
 'replaced_modules': [],
 'final_alphas': {},
 'final_saturation': {},
 'dyt_alpha_init': 0.5,
 'dyt_preserve_ln_affine': False}

### Run `vit_dyt_full_fresh` — classification, DyT with paper-default init (γ=1, β=0, α₀=0.5)

In [16]:
train_one(cfg_classification(variant="dyt_full", run_name="vit_dyt_full_fresh"), RUNS_DIR)


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([100]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([100, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[replace_ln_with_dyt] preserve_affine=False replaced 25 LayerNorms under 'vit.'
[vit_dyt_full_fresh] step 50/3000  loss=4.8690  lr=1.22e-05  elapsed=5s
[vit_dyt_full_fresh] step 100/3000  loss=4.7608  lr=2.48e-05  elapsed=10s
[vit_dyt_full_fresh] step 150/3000  loss=4.6677  lr=3.72e-05  elapsed=15s
[vit_dyt_full_fresh] step 200/3000  loss=4.6446  lr=4.98e-05  elapsed=20s
[vit_dyt_full_fresh] step 250/3000  loss=4.6420  lr=5.00e-05  elapsed=25s
[vit_dyt_full_fresh] step 300/3000  loss=4.5889  lr=4.98e-05  elapsed=30s
[vit_dyt_full_fresh] step 350/3000  loss=4.6113  lr=4.97e-05  elapsed=34s
[vit_dyt_full_fresh] step 400/3000  loss=4.5771  lr=4.94e-05  elapsed=39s
[vit_dyt_full_fresh] step 450/3000  loss=4.5984  lr=4.90e-05  elapsed=44s
[vit_dyt_full_fresh] step 500/3000  loss=4.6309  lr=4.86e-05  elapsed=49s
[vit_dyt_full_fresh] eval @ 500: top1=0.0169
[vit_dyt_full_fresh] step 550/3000  loss=4.5881  lr=4.81e-05  elapsed=65s
[vit_dyt_full_fresh] step 600/3000  loss=4.5787  lr=4.75e-05  e

{'run_name': 'vit_dyt_full_fresh',
 'task': 'classification',
 'variant': 'dyt_full',
 'data_source': 'uoft-cs/cifar100',
 'iters': 3000,
 'best_primary': 0.045,
 'metric_name': 'top1',
 'num_replaced': 25,
 'replaced_modules': ['vit.encoder.layer.0.layernorm_before',
  'vit.encoder.layer.0.layernorm_after',
  'vit.encoder.layer.1.layernorm_before',
  'vit.encoder.layer.1.layernorm_after',
  'vit.encoder.layer.2.layernorm_before',
  'vit.encoder.layer.2.layernorm_after',
  'vit.encoder.layer.3.layernorm_before',
  'vit.encoder.layer.3.layernorm_after',
  'vit.encoder.layer.4.layernorm_before',
  'vit.encoder.layer.4.layernorm_after',
  'vit.encoder.layer.5.layernorm_before',
  'vit.encoder.layer.5.layernorm_after',
  'vit.encoder.layer.6.layernorm_before',
  'vit.encoder.layer.6.layernorm_after',
  'vit.encoder.layer.7.layernorm_before',
  'vit.encoder.layer.7.layernorm_after',
  'vit.encoder.layer.8.layernorm_before',
  'vit.encoder.layer.8.layernorm_after',
  'vit.encoder.layer.9.lay

### Run `vit_dyt_full_affine` — classification, DyT with affine-preserving init (γ←γ_LN, β←β_LN, α₀=0.5)

In [17]:
train_one(cfg_classification(variant="dyt_full", run_name="vit_dyt_full_affine", dyt_preserve_ln_affine=True), RUNS_DIR)


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([100]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([100, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[replace_ln_with_dyt] preserve_affine=True replaced 25 LayerNorms under 'vit.'
[vit_dyt_full_affine] step 50/3000  loss=4.6140  lr=1.22e-05  elapsed=5s
[vit_dyt_full_affine] step 100/3000  loss=4.6371  lr=2.48e-05  elapsed=10s
[vit_dyt_full_affine] step 150/3000  loss=4.5607  lr=3.72e-05  elapsed=15s
[vit_dyt_full_affine] step 200/3000  loss=4.4492  lr=4.98e-05  elapsed=20s
[vit_dyt_full_affine] step 250/3000  loss=4.3066  lr=5.00e-05  elapsed=24s
[vit_dyt_full_affine] step 300/3000  loss=4.0328  lr=4.98e-05  elapsed=29s
[vit_dyt_full_affine] step 350/3000  loss=4.1167  lr=4.97e-05  elapsed=34s
[vit_dyt_full_affine] step 400/3000  loss=3.9954  lr=4.94e-05  elapsed=39s
[vit_dyt_full_affine] step 450/3000  loss=3.9557  lr=4.90e-05  elapsed=44s
[vit_dyt_full_affine] step 500/3000  loss=3.9919  lr=4.86e-05  elapsed=49s
[vit_dyt_full_affine] eval @ 500: top1=0.1399
[vit_dyt_full_affine] step 550/3000  loss=3.8492  lr=4.81e-05  elapsed=65s
[vit_dyt_full_affine] step 600/3000  loss=3.3350  lr

{'run_name': 'vit_dyt_full_affine',
 'task': 'classification',
 'variant': 'dyt_full',
 'data_source': 'uoft-cs/cifar100',
 'iters': 3000,
 'best_primary': 0.4832,
 'metric_name': 'top1',
 'num_replaced': 25,
 'replaced_modules': ['vit.encoder.layer.0.layernorm_before',
  'vit.encoder.layer.0.layernorm_after',
  'vit.encoder.layer.1.layernorm_before',
  'vit.encoder.layer.1.layernorm_after',
  'vit.encoder.layer.2.layernorm_before',
  'vit.encoder.layer.2.layernorm_after',
  'vit.encoder.layer.3.layernorm_before',
  'vit.encoder.layer.3.layernorm_after',
  'vit.encoder.layer.4.layernorm_before',
  'vit.encoder.layer.4.layernorm_after',
  'vit.encoder.layer.5.layernorm_before',
  'vit.encoder.layer.5.layernorm_after',
  'vit.encoder.layer.6.layernorm_before',
  'vit.encoder.layer.6.layernorm_after',
  'vit.encoder.layer.7.layernorm_before',
  'vit.encoder.layer.7.layernorm_after',
  'vit.encoder.layer.8.layernorm_before',
  'vit.encoder.layer.8.layernorm_after',
  'vit.encoder.layer.9.l

## 10. Diagnostics

In [ ]:
import matplotlib.pyplot as plt

DISPLAY_NAME = {
    "vit_ln": "LayerNorm",
    "seg_ln": "LayerNorm",
    "vit_dyt_full_fresh": "DyT default",
    "seg_dyt_full_fresh": "DyT default",
    "vit_dyt_full_affine": "DyT affine",
    "seg_dyt_full_affine": "DyT affine",
}
COLOR = {
    "LayerNorm": "tab:blue",
    "DyT default": "tab:green",
    "DyT affine": "tab:orange",
}
ORDER = {"LayerNorm": 0, "DyT default": 1, "DyT affine": 2}


def _read_jsonl(p):
    with open(p) as f:
        return [json.loads(line) for line in f]


def _runs_for_task(task):
    runs = [d.name for d in RUNS_DIR.iterdir() if d.is_dir() and (d / "summary.json").exists()
            and json.loads((d / "summary.json").read_text()).get("task") == task]
    return sorted(runs, key=lambda r: ORDER.get(DISPLAY_NAME.get(r, r), 99))


def plot_curves(task, out_path):
    fig_l, ax_l = plt.subplots(figsize=(5, 3))
    fig_m, ax_m = plt.subplots(figsize=(5, 3))
    metric_name = "miou" if task == "segmentation" else "top1"
    for run in _runs_for_task(task):
        log = _read_jsonl(RUNS_DIR / run / "log.jsonl")
        train_rows = [r for r in log if r["kind"] == "train"]
        eval_rows = [r for r in log if r["kind"] == "eval"]
        name = DISPLAY_NAME.get(run, run)
        c = COLOR.get(name)
        ax_l.plot([r["step"] for r in train_rows], [r["loss"] for r in train_rows], label=name, color=c)
        ax_m.plot([r["step"] for r in eval_rows], [r["primary"] for r in eval_rows], marker="o", label=name, color=c)
    for ax, ttl, ylab in [(ax_l, f"{task} training loss", "loss"), (ax_m, f"{task} validation {metric_name}", metric_name)]:
        ax.set_xlabel("iter"); ax.set_ylabel(ylab); ax.set_title(ttl); ax.legend(fontsize=7); ax.grid(alpha=0.3)
    fig_l.tight_layout(); fig_m.tight_layout()
    fig_l.savefig(out_path / f"{task}_loss.pdf")
    fig_m.savefig(out_path / f"{task}_{metric_name}.pdf")


def plot_alpha_drift(task, out_path):
    fig, ax = plt.subplots(figsize=(8, 3))
    for run in _runs_for_task(task):
        s = json.loads((RUNS_DIR / run / "summary.json").read_text())
        alphas = s.get("final_alphas", {})
        if not alphas: continue
        names = list(alphas.keys())
        vals = [alphas[n] for n in names]
        name = DISPLAY_NAME.get(run, run)
        ax.plot(range(len(vals)), vals, marker=".", label=name, color=COLOR.get(name))
    ax.axhline(0.5, color="grey", linestyle="--", linewidth=0.7, label="α₀=0.5")
    ax.set_xlabel("DyT layer index"); ax.set_ylabel("trained α")
    ax.set_title(f"{task}: trained α per layer"); ax.legend(fontsize=7); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(out_path / f"{task}_alpha.pdf")


def plot_saturation(task, out_path):
    fig, ax = plt.subplots(figsize=(8, 3))
    for run in _runs_for_task(task):
        s = json.loads((RUNS_DIR / run / "summary.json").read_text())
        sat = s.get("final_saturation", {})
        if not sat: continue
        names = list(sat.keys())
        vals = [sat[n] for n in names]
        name = DISPLAY_NAME.get(run, run)
        ax.plot(range(len(vals)), vals, marker=".", label=name, color=COLOR.get(name))
    ax.set_xlabel("DyT layer index"); ax.set_ylabel("frac |tanh(αx)| > 0.95")
    ax.set_title(f"{task}: saturation per layer"); ax.legend(fontsize=7); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(out_path / f"{task}_saturation.pdf")


for task in ["segmentation", "classification"]:
    plot_curves(task, FIGURES_DIR)
    plot_alpha_drift(task, FIGURES_DIR)
    plot_saturation(task, FIGURES_DIR)


## 11. Summary table

In [28]:
rows = []
for d in sorted(RUNS_DIR.iterdir()):
    p = d / "summary.json"
    if p.exists():
        s = json.loads(p.read_text())
        rows.append({
            "run": s.get("run_name", d.name),
            "task": s.get("task"),
            "variant": s.get("variant"),
            "preserve_affine": s.get("dyt_preserve_ln_affine"),
            "metric": s.get("metric_name"),
            "best": round(s.get("best_primary", -1), 4),
            "n_replaced": s.get("num_replaced"),
        })
print(json.dumps(rows, indent=2))
(FIGURES_DIR / "table_main.json").write_text(json.dumps(rows, indent=2))


[
  {
    "run": "seg_dyt_full_affine",
    "task": "segmentation",
    "variant": "dyt_full",
    "preserve_affine": true,
    "metric": "miou",
    "best": 0.0099,
    "n_replaced": 30
  },
  {
    "run": "seg_dyt_full_fresh",
    "task": "segmentation",
    "variant": "dyt_full",
    "preserve_affine": false,
    "metric": "miou",
    "best": 0.0138,
    "n_replaced": 30
  },
  {
    "run": "seg_ln",
    "task": "segmentation",
    "variant": "ln",
    "preserve_affine": false,
    "metric": "miou",
    "best": 0.0765,
    "n_replaced": 0
  },
  {
    "run": "vit_dyt_full_affine",
    "task": "classification",
    "variant": "dyt_full",
    "preserve_affine": true,
    "metric": "top1",
    "best": 0.4832,
    "n_replaced": 25
  },
  {
    "run": "vit_dyt_full_fresh",
    "task": "classification",
    "variant": "dyt_full",
    "preserve_affine": false,
    "metric": "top1",
    "best": 0.045,
    "n_replaced": 25
  },
  {
    "run": "vit_ln",
    "task": "classification",
    "vari

1109